In [2]:
import dashscope
from dashscope import TextEmbedding
import numpy as np
import faiss
import pickle
from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
import os
# 1. 加载环境变量
load_dotenv()
dashscope.api_key = os.getenv("DASHSCOPE_API_KEY")
# 2. 配置模型 (env里面同时配置了阿里和openai的apikey 相同)
model = OpenAIChatModel(
    model_name='qwen-plus',
)

In [3]:
def embed(texts):
    if isinstance(texts, str):
        texts = [texts]

    resp = TextEmbedding.call(
        model="text-embedding-v2",
        input=texts
    )

    return [item["embedding"] for item in resp["output"]["embeddings"]]

In [33]:
import os

def load_txt_files(folder_path):
    docs = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder_path, filename)

            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()
                docs.append(text)

    return docs
docs = load_txt_files(r"E:\learn\projectNewENV")


def split_with_overlap(text, chunk_size=10, overlap=3):
    #chunk_size每10个字符切一次  overlap每一个块重复3个字符 这样避免上下文丢失
    chunks = []
    start = 0
    i = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        print(f"\n=== chunk {i} ===")
        print(f"start={start}, end={end}")
        print(chunk)# 看看切出来的东西
        start += chunk_size - overlap
        i+=1
    return chunks




=== chunk 0 ===
start=0, end=10
公司年假政策：
员工

=== chunk 1 ===
start=7, end=17

员工工作满 1 年

=== chunk 2 ===
start=14, end=24
1 年可享受 5 天

=== chunk 3 ===
start=21, end=31
5 天带薪年假
年假

=== chunk 4 ===
start=28, end=38

年假有效期至次年 

=== chunk 5 ===
start=35, end=45
次年 3 月 31 

=== chunk 6 ===
start=42, end=52
31 日
部门经理审

=== chunk 7 ===
start=49, end=59
经理审批后方可使用


=== chunk 8 ===
start=56, end=66
使用
考勤制度：
工

=== chunk 9 ===
start=63, end=73
：
工作日上班时间：

=== chunk 10 ===
start=70, end=80
时间：9:00-18

=== chunk 11 ===
start=77, end=87
-18:00
迟到超

=== chunk 12 ===
start=84, end=94
迟到超过 30 分钟

=== chunk 13 ===
start=91, end=101
 分钟记为缺勤


In [12]:
vectors = embed(docs)

vectors_np = np.array(vectors).astype("float32")

dim = vectors_np.shape[1]
index = faiss.IndexFlatL2(dim)

index.add(vectors_np)

print("向量库构建完成")

向量库构建完成


In [13]:
faiss.write_index(index, "faiss.index") #保存为faiss 作为页码

with open("docs.pkl", "wb") as f: #实际保存的正文
    pickle.dump(docs, f)

print("保存成功")

保存成功


In [14]:
index = faiss.read_index("faiss.index") # 用faiss 读取之前保存的向量数据加载

with open("docs.pkl", "rb") as f:
    docs = pickle.load(f) #用pickle模块读取二进制

print("加载成功")

加载成功


In [1]:
query = "员工超过9点上班算不算迟到"

query_vec = np.array(embed(query)).astype("float32")

D, I = index.search(query_vec, k=2)

for idx in I[0]:
    print(docs[idx])

NameError: name 'np' is not defined